In [ ]:
# Data Preprocessing Pipeline

from pathlib import Path
import random
import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms

# Reproducibility

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random seed fixed: {SEED}")


# Project Paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR = DATA_DIR / "test"

METADATA_PATH = DATA_DIR / "metadata.csv"

print(f"Project Root : {PROJECT_ROOT}")
print(f"Training Data: {TRAIN_DIR}")
print(f"Validation Data: {VAL_DIR}")
print(f"Testing Data : {TEST_DIR}")

In [ ]:
# Verify required files and folders

required_paths = [
    TRAIN_DIR,
    VAL_DIR,
    TEST_DIR,
    METADATA_PATH,
]

for path in required_paths:
    if path.exists():
        print(f"✓ {path}")
    else:
        print(f"✗ Missing: {path}")

if METADATA_PATH.exists():
    metadata = pd.read_csv(METADATA_PATH)
    print(f"\nMetadata shape: {metadata.shape}")
    display(metadata.head())

In [ ]:
# Image Preprocessing & Data Augmentation

IMAGE_SIZE = 224

# ImageNet statistics (recommended for transfer learning models)
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

print(" Image preprocessing transforms created successfully.")

In [ ]:
# Create ImageFolder Datasets

train_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=train_transforms
)

val_dataset = datasets.ImageFolder(
    root=VAL_DIR,
    transform=val_transforms
)

test_dataset = datasets.ImageFolder(
    root=TEST_DIR,
    transform=test_transforms
)

print("Datasets loaded successfully.\n")

print(f"Training images   : {len(train_dataset)}")
print(f"Validation images : {len(val_dataset)}")
print(f"Testing images    : {len(test_dataset)}")

print("\nClass Mapping:")
print(train_dataset.class_to_idx)

In [ ]:
# Check Class Distribution

from collections import Counter

train_labels = [label for _, label in train_dataset.samples]

class_counts = Counter(train_labels)

print("Training Class Distribution")
print("-" * 35)

for class_name, class_idx in train_dataset.class_to_idx.items():
    print(f"{class_name:<10}: {class_counts[class_idx]} images")

total_images = len(train_dataset)

class_weights = {
    cls: total_images / (len(class_counts) * count)
    for cls, count in class_counts.items()
}


print("\nComputed Class Weights")
print(class_weights)

In [ ]:
# Create Weighted Random Sampler


sample_weights = [
    class_weights[label]
    for _, label in train_dataset.samples
]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("WeightedRandomSampler created successfully.")

In [ ]:
# Create PyTorch DataLoaders

BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print(" DataLoaders created successfully")
print()
print(f"Train batches      : {len(train_loader)}")
print(f"Validation batches : {len(val_loader)}")
print(f"Test batches       : {len(test_loader)}")

In [ ]:
# Visualize a Batch of Preprocessed Images

import matplotlib.pyplot as plt

# Get one batch
images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch labels shape:", labels.shape)

# Display first 8 images
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for idx, ax in enumerate(axes.flat):
    image = images[idx].permute(1, 2, 0)

    # Undo normalization for visualization
    image = image * torch.tensor(STD) + torch.tensor(MEAN)
    image = torch.clamp(image, 0, 1)

    ax.imshow(image)
    ax.set_title(
        list(train_dataset.class_to_idx.keys())[labels[idx]]
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Verify DataLoader output

import torch

images, labels = next(iter(train_loader))

print("Batch verification")
print("------------------")
print("Images shape :", images.shape)
print("Labels shape :", labels.shape)

print("\nImage tensor details")
print("-------------------")
print("Data type :", images.dtype)
print("Min pixel :", images.min().item())
print("Max pixel :", images.max().item())

print("\nLabel distribution in batch")
print("---------------------------")
unique, counts = torch.unique(labels, return_counts=True)

for label, count in zip(unique.tolist(), counts.tolist()):
    print(f"Class {label}: {count} images")

In [ ]:
PREPROCESSING_CONFIG = {
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_classes": len(train_dataset.classes),
    "classes": train_dataset.classes,
    "normalization": {
        "mean": MEAN,
        "std": STD
    },
    "augmentation": [
        "RandomHorizontalFlip",
        "RandomRotation(10)",
        "ColorJitter(brightness=0.1, contrast=0.1)"
    ],
    "class_balancing": "WeightedRandomSampler"
}

PREPROCESSING_CONFIG

In [ ]:
print("Preprocessing Pipeline Verification")
print("-" * 50)

print(f"Classes: {train_dataset.classes}")
print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch size: {BATCH_SIZE}")

print("\nDataset Sizes")
print("-" * 50)
print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

print("\nPipeline status: READY FOR MODEL TRAINING")